In [1]:
# imports
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder , OneHotEncoder
from sklearn.feature_selection import chi2 , SelectKBest
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from imblearn.over_sampling import RandomOverSampler
from sklearn.ensemble import RandomForestClassifier
# Construct the pipeline
from sklearn.pipeline import Pipeline
import pickle


In [2]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ml_projects/accident_prediction_system/accident_dataset.csv')
df2 = df.copy()
# df2.shape

In [3]:
df2['Time'] = pd.to_datetime(df2['Time'])
df2['hour_of_the_day'] = df2['Time'].dt.hour
new_df = df2.copy()
new_df.drop('Time',axis=1,inplace = True)
# new_df.head()

/tmp/ipython-input-3097468963.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df2['Time'] = pd.to_datetime(df2['Time'])


DATA ENCODING OF OUTPUT COLUMN

In [4]:
# output data label encoding
le = LabelEncoder()
new_df['Accident_severity'] = le.fit_transform(new_df['Accident_severity'])
new_df['Accident_severity'].value_counts()

,count
Accident_severity,
2,10415
1,1743
0,158


BALANCE OUTPUT COLUMN DATA

In [5]:
# data balance
x = new_df.drop('Accident_severity',axis=1)
y = new_df['Accident_severity']
oversampler = RandomOverSampler(random_state= 42)
x_resampled , y_resampled = oversampler.fit_resample(x,y)
# y_resampled.value_counts()


TRAIN , TEST , SPLIT

In [6]:
x_train , x_test , y_train , y_test = train_test_split(x_resampled,y_resampled,test_size=0.2, random_state = 42)
# x_train.shape , x_test.shape , y_train.shape , y_test.shape
x_train.isnull().sum()

,0
Day_of_week,0
Age_band_of_driver,0
Sex_of_driver,0
Educational_level,1755
Vehicle_driver_relation,1312
Driving_experience,1900
Type_of_vehicle,1872
Owner_of_vehicle,949
Service_year_of_vehicle,7443
Defect_of_vehicle,8843


Use of Column Transfer pipeline for the :
1. fill missing values,
2. encode the categorical columns,
3. scale the input column values,
4. feature selection using chi2 statistics ,

1. Filling missing values transformaer

In [7]:
# Define the strategies for each column
strategies = {
    1: 'most_frequent',   # Educational_level , Vehicle_driver_relation , Driving_experience , Type_of_vehicle , Area_accident_occured,Fitness_of_casuality,Work_of_casuality,Vehicle_movement,Type_of_collision,Road_surface_type,Types_of_Junction,Road_allignment,Lanes_or_Medians
    2: 'constant',        # Service_year_of_vehicle , Defect_of_vehicle
}

# Create a ColumnTransformer for data preprocessing
tf1 = ColumnTransformer([
    ('impute_educational_level', SimpleImputer(strategy=strategies[1]), [3]),
    ('impute_Vehicle_driver_relation', SimpleImputer(strategy=strategies[1]), [4]),
    ('impute_Driving_experience', SimpleImputer(strategy=strategies[1]), [5]),
    ('impute_Type_of_vehicle', SimpleImputer(strategy=strategies[1]), [6]),
    ('impute_Service_year_of_vehicle', SimpleImputer(strategy=strategies[2], fill_value='Unknown'), [8]),
    ('impute_Defect_of_vehicle', SimpleImputer(strategy=strategies[2], fill_value='Unknown'), [9]),
    ('impute_Area_accident_occured', SimpleImputer(strategy=strategies[1]), [10]),
    ('impute_Lanes_or_Medians', SimpleImputer(strategy=strategies[1]), [11]),
    ('impute_Road_allignment', SimpleImputer(strategy=strategies[1]), [12]),
    ('impute_Types_of_Junction', SimpleImputer(strategy=strategies[1]), [13]),
    ('impute_Road_surface_type', SimpleImputer(strategy=strategies[1]), [14]),
    ('impute_Type_of_collision', SimpleImputer(strategy=strategies[1]), [18]),
    ('impute_Vehicle_movement', SimpleImputer(strategy=strategies[1]), [21]),
    ('impute_Work_of_casuality', SimpleImputer(strategy=strategies[1]), [26]),
    ('impute_Fitness_of_casuality', SimpleImputer(strategy=strategies[1]), [27])
], remainder='passthrough')


In [8]:
tf1

ColumnTransformer(remainder='passthrough',
                  transformers=[('impute_educational_level',
                                 SimpleImputer(strategy='most_frequent'), [3]),
                                ('impute_Vehicle_driver_relation',
                                 SimpleImputer(strategy='most_frequent'), [4]),
                                ('impute_Driving_experience',
                                 SimpleImputer(strategy='most_frequent'), [5]),
                                ('impute_Type_of_vehicle',
                                 SimpleImputer(strategy='most_frequent'), [6...
                                ('impute_Road_surface_type',
                                 SimpleImputer(strategy='most_frequent'),
                                 [14]),
                                ('impute_Type_of_collision',
                                 SimpleImputer(strategy='most_frequent'),
                                 [18]),
                                ('impute_Vehicle_movement',
                                 SimpleImputer(strategy='most_frequent'),
                                 [21]),
                                ('impute_Work_of_casuality',
                                 SimpleImputer(strategy='most_frequent'),
                                 [26]),
                                ('impute_Fitness_of_casuality',
                                 SimpleImputer(strategy='most_frequent'),
                                 [27])])

2. Encode the categorical columns transformaer

In [9]:
# define the object column indices
obj_col_indices = []
for i in range(31):
  obj_col_indices.append(i)

tf2 = ColumnTransformer([
    (f'ohe_{col}', OneHotEncoder(sparse_output = False, handle_unknown='ignore'), [col])
    for col in  obj_col_indices
], remainder='passthrough')

x_train_encoded = tf2.fit_transform(x_train)
x_test_encoded = tf2.transform(x_test)
x_train_encoded.shape

(24996, 258)

In [10]:
tf2

ColumnTransformer(remainder='passthrough',
                  transformers=[('ohe_0',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 [0]),
                                ('ohe_1',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 [1]),
                                ('ohe_2',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 [2]),
                                ('ohe_3',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 [3]),
                                ('...
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 [25]),
                                ('ohe_26',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 [26]),
                                ('ohe_27',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 [27]),
                                ('ohe_28',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 [28]),
                                ('ohe_29',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 [29]), ...])

3. Scale the input column values (using chi2)

chi2: This is one of the scoring functions available for feature selection in scikit-learn. It calculates the chi-squared statistic between each feature and the target variable (accidents) to determine the relevance of each feature. chi2 is commonly used for feature selection when dealing with categorical target variables.

In [12]:
tf4 = SelectKBest(chi2, k=50)

SelectKBest(k=50, score_func=<function chi2 at 0x7fe3e27513a0>)

##  **Model (Random Forest Classifier)**

In [13]:
tf5 = RandomForestClassifier()

In [17]:
# Create the pipeline
pipe = Pipeline([
    ('tf1', tf1),
    ('tf2', tf2),
    ('tf4', tf4),
    ('tf5', tf5)
])

# pipe line taining
pipe.fit(x_train , y_train)

Pipeline(steps=[('tf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_educational_level',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [3]),
                                                 ('impute_Vehicle_driver_relation',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [4]),
                                                 ('impute_Driving_experience',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [5]),
                                                 ('impute_Type_of_vehicle',
                                                  SimpleImputer(strat...
                                                                sparse_output=False),
                                                  [26]),
                                                 ('ohe_27',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [27]),
                                                 ('ohe_28',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [28]),
                                                 ('ohe_29',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [29]), ...])),
                ('tf4',
                 SelectKBest(k=50,
                             score_func=<function chi2 at 0x7fe3e27513a0>)),
                ('tf5', RandomForestClassifier())])

In [23]:
# Predict
y_pred = pipe.predict(x_test)
y_pred

array([1, 2, 2, ..., 0, 1, 2])

In [26]:
# accuracy check
accuracy_score(y_test,y_pred)

0.9228676588254121

## Classificatoin report

In [27]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99      2085
           1       0.85      0.95      0.90      2100
           2       0.95      0.82      0.88      2064

    accuracy                           0.92      6249
   macro avg       0.93      0.92      0.92      6249
weighted avg       0.93      0.92      0.92      6249



##Confusion matrics

In [29]:
confusion_matrix(y_test,y_pred)

array([[2085,    0,    0],
       [  22, 1997,   81],
       [  29,  350, 1685]])

Save the pipe

In [33]:
pickle.dump(pipe,open("accident_prediction_trained_model_pipe.pkl",'wb'))

Testing

In [36]:
#testing (record from X_test 10 row)
print("Prediction :",pipe.predict(np.array(['Thursday', '31-50', 'Male', 'Junior high school', 'Owner', 'Unknown', 'Long lorry', 'Owner',
                       'Unknown', 'Unknown', 'Other', 'Two-way (divided with solid lines road marking)',
                       'Tangent road with flat terrain','Unknown', 'Unknown', 'Dry', 'Daylight', 'Normal',
                       'Collision with animals', 2, 1, 'Going straight', 'Driver or rider','Male', '18-30', 3, 'Driver',
                       'Normal', 'Not a Pedestrian', 'Changing lane to the left', 12,],dtype=object).reshape(1,-1)))
print("Actual :",y_test.iloc[10])

Prediction : [1]
Actual : 1


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

function to execute

In [38]:
def pred(Day_of_week, Age_band_of_driver, Sex_of_driver, Educational_level, Vehicle_driver_relation,
         Driving_experience, Type_of_vehicle, Owner_of_vehicle, Service_year_of_vehicle,
         Defect_of_vehicle, Area_accident_occured, Lanes_or_Medians, Road_allignment,
         Types_of_Junction, Road_surface_type, Road_surface_conditions, Light_conditions,
         Weather_conditions, Type_of_collision, Number_of_vehicles_involved,
         Number_of_casualties, Vehicle_movement, Casualty_class, Sex_of_casualty,
         Age_band_of_casualty, Casualty_severity, Work_of_casuality, Fitness_of_casuality,
         Pedestrian_movement, Cause_of_accident, Hour_of_Day):

    # Your prediction code here
    features = np.array([[Day_of_week, Age_band_of_driver, Sex_of_driver, Educational_level, Vehicle_driver_relation,
         Driving_experience, Type_of_vehicle, Owner_of_vehicle, Service_year_of_vehicle,
         Defect_of_vehicle, Area_accident_occured, Lanes_or_Medians, Road_allignment,
         Types_of_Junction, Road_surface_type, Road_surface_conditions, Light_conditions,
         Weather_conditions, Type_of_collision, Number_of_vehicles_involved,
         Number_of_casualties, Vehicle_movement, Casualty_class, Sex_of_casualty,
         Age_band_of_casualty, Casualty_severity, Work_of_casuality, Fitness_of_casuality,
         Pedestrian_movement, Cause_of_accident, Hour_of_Day]])

    results = pipe.predict(features)
    if results[0] == 2:
      return "Slight Injury"
    elif results[0] == 1:
      return "Serious Injury"
    else:
      return "Fatal Injury"

predicted_class = pred(Day_of_week="Friday",
                       Age_band_of_driver='31-50',
                       Sex_of_driver='Male',
                       Educational_level='Elementary school',
                       Vehicle_driver_relation='Employee',
                       Driving_experience='1-2yr',
                       Type_of_vehicle='Lorry (41?100Q)',
                       Owner_of_vehicle='Owner',
                       Service_year_of_vehicle=None,
                       Defect_of_vehicle='No defect',
                       Area_accident_occured='Office areas',
                       Lanes_or_Medians='Two-way (divided with broken lines road marking)',
                       Road_allignment='Tangent road with flat terrain',
                       Types_of_Junction='Y Shape',
                       Road_surface_type='Asphalt roads',
                       Road_surface_conditions='Dry',
                       Light_conditions='Daylight',
                       Weather_conditions='Normal',
                       Type_of_collision='Vehicle with vehicle collision',
                       Number_of_vehicles_involved=2,
                       Number_of_casualties=2,
                       Vehicle_movement='Going straight',
                       Casualty_class='na',
                       Sex_of_casualty='na',
                       Age_band_of_casualty='na',
                       Casualty_severity='na',
                       Work_of_casuality='Driver',
                       Fitness_of_casuality='Normal',
                       Pedestrian_movement='Not a Pedestrian',
                       Cause_of_accident='Changing lane to the left',
                       Hour_of_Day=1)
print(predicted_class)

Slight Injury


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v